# Vision QA — inventarul imaginilor de piese

**Scop.** Verifică datasetul de clasificare din S3: câte piese, câte imagini, ce clase, cum e
împărțit pe familii și dacă splitul ține aceeași piesă într-un singur set.

**Intrare.** `s3://cncdt-dev-training-data-864981730944/training-data/parts/dt=2026-10-18/run-20261018T201942/`.
**Ieșire.** niciuna; notebook-ul confirmă datele dinainte de antrenare.

Etichetele (`defect`, `verdict`, `family`) rămân în index și în rapoarte. Modelul primește doar
imaginea.



## 1. Importuri și căi


In [ ]:
import io
import json

import boto3
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

CORPUS = "cncdt-dev-training-data-864981730944"
PARTS = "training-data/parts/dt=2026-10-18/run-20261018T201942/"
s3 = boto3.client("s3")

manifest = json.loads(s3.get_object(Bucket=CORPUS, Key=PARTS + "manifest.json")["Body"].read())
print(json.dumps({k: manifest[k] for k in ("dataset", "totals", "verdicts", "splits")},
                 indent=1, ensure_ascii=False))



## 2. Indexurile celor trei splituri


In [ ]:
def read_index(name):
    body = s3.get_object(Bucket=CORPUS, Key=PARTS + name)["Body"].read().decode()
    return pd.DataFrame([json.loads(line) for line in body.splitlines() if line])


train = read_index("images-train.jsonl")
val = read_index("images-val.jsonl")
test = read_index("images-test.jsonl")

for name, frame in (("train", train), ("val", val), ("test", test)):
    print(name, len(frame), "imagini,", frame["part_id"].nunique(), "piese,",
          frame["verdict"].value_counts().to_dict())



## 3. Scurgere între splituri

Splitul se face pe `part_id`, nu pe imagine. Verificarea de mai jos trebuie să dea zero
peste tot; altfel aceeași piesă ajunge în două seturi și metricele se umflă.



In [ ]:
parts = {name: set(frame["part_id"]) for name, frame in
         (("train", train), ("val", val), ("test", test))}
print("train ∩ val:", len(parts["train"] & parts["val"]))
print("train ∩ test:", len(parts["train"] & parts["test"]))
print("val ∩ test:", len(parts["val"] & parts["test"]))



## 4. Echilibrul claselor


In [ ]:
display(train.groupby(["family", "verdict"]).size().unstack(fill_value=0))
display(train["defect"].value_counts().to_frame("piese"))
print("PASS din total (imagini):", round((train["verdict"] == "PASS").mean(), 3))



## 5. Câteva imagini


In [ ]:
sample = pd.concat([
    train[train["verdict"] == "PASS"].drop_duplicates("part_id").head(3),
    train[train["verdict"] == "REJECT"].drop_duplicates("part_id").head(3),
])

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, (_, row) in zip(axes.ravel(), sample.iterrows()):
    body = s3.get_object(Bucket=CORPUS, Key=PARTS + row["path"])["Body"].read()
    ax.imshow(Image.open(io.BytesIO(body)))
    ax.axis("off")
    ax.set_title(f"{row['family']} · {row['defect']} · {row['verdict']}", fontsize=8)
fig.tight_layout()



## 6. Ce urmează

Clasele sunt dezechilibrate: ~13% PASS în antrenare. Antrenarea folosește greutăți pe clase și
raportează recall separat pe PASS și REJECT. Deschide `qa-02-vision.ipynb`.

